# DBS outcome — complete reproduction

Everything in the DBS result, from the cached volumes to the external metric table and
the verification tests. No stored predictions are read: every number is recomputed here.

**Cohorts.** MSW (internal) 66 patients, 61 responders / 5 non-responders, evaluated
leave-one-out. CHH (external) 37 patients, 34 / 3, touched once at the end.

**Model.** The K lowest Fourier frequencies of both nuclei, log-magnitude, each
coefficient one token, concatenated with 7 clinical covariate tokens; one self-attention
layer; mean-pool to a linear head; plus a linear residual on the raw vector.

**Capacity selection.** Every model's capacity parameter is chosen by internal
leave-one-out AUC alone, under the same rule. The external cohort plays no part.

**Runtime.** Roughly an hour on the first run. Both slow stages cache to
`docs/dbs_3d/`, so re-running is fast; delete those files to force a recompute.

In [1]:
import sys, os, re, time, pickle, warnings
warnings.filterwarnings('ignore')
REPO, SPDIR = '/workspace', '/workspace/dbs_3d'
sys.path.insert(0, REPO); sys.path.insert(0, SPDIR); os.chdir(REPO)
import numpy as np, pandas as pd, torch
from scipy.stats import rankdata
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score, f1_score,
                             accuracy_score, confusion_matrix)

NSEED, KGRID = 10, [8, 16, 32]
# Order is load-bearing: the pre-specified test draws from one sequential RNG across
# this list. Appending is safe; reordering silently changes the p-values of every model
# after the insertion point.
BASE_MODELS = ['SpectralViT', 'Spectral LR', 'Spectral MLP', 'Clinical', 'Spatial ViT']
MODELS = BASE_MODELS + ['Attention UNet']
assert MODELS[:len(BASE_MODELS)] == BASE_MODELS, 'appending only -- see note above'
DATA  = os.environ.get('DBS_CLINICAL_DIR', os.path.join(REPO, 'docs'))
CACHE = os.environ.get('DBS_CACHE_DIR', '/checkpoints')

# sc_worker.py holds the data prep, the feature construction and the models. Split it
# at the K-dependent line so the cohort loads once and only the features rebuild per K.
# Note line 32 of that file contains the same marker inside a string literal, so the
# split has to be on the call site.
SRC = open(os.path.join(SPDIR, 'sc_worker.py')).read().split('t0 = time.time()')[0]
MARK = chr(10) + 'IDX = low_modes('
HEAD, TAIL = SRC.split(MARK, 1); TAIL = 'IDX = low_modes(' + TAIL

G = {'__file__': os.path.join(SPDIR, 'sc_worker.py')}   # sc_worker derives SPDIR from it
_a = sys.argv; sys.argv = [_a[0], 'cell', 'SpectralViT', '16', '1']
try:
    exec(compile(HEAD, 'sc_worker_head', 'exec'), G)
    HAVE_DATA = True
except Exception as e:
    HAVE_DATA = False
    print('Patient data missing -- the cohort cache or clinical table is not available here.')
    print(f'  cache    {CACHE}')
    print(f'  clinical {DATA}')
    print(f'  ({type(e).__name__}: {e})')
finally:
    sys.argv = _a

if HAVE_DATA:
    bA, bB = G['bA'], G['bB']
    print()
    print(f"MSW {len(bA)}  {int(bA.sum())} resp / {int((bA==0).sum())} non-resp")
    print(f"CHH {len(bB)}  {int(bB.sum())} resp / {int((bB==0).sum())} non-resp")
    print('device:', G['dev'])


Preparing MSW dataset (Forced 6-dim alignment) 
Pre-flight check: Validating MSW CSV mapping...
--- MSW CSV RAW MEANS ---
  > Age     : 62.13
  > Sex     : 0.26
  > Dur     : 8.47
  > LEDD    : 989.80
  > Off-Pre : 45.91
  > On-Pre  : 19.60
Cache present: 109 subjects, skipping raw NIfTI discovery.

Final MSW Breakdown:
 - Unique Subjects on Disk: 109
 - Labeled Responders (1): 61
 - Labeled Non-Responders (0): 5
 - Unlabeled subjects (-1): 43
 - Verified Realized Means (Matched Data Only):
    > Age     : 63.08
    > Sex     : 0.26
    > Dur     : 8.44
    > LEDD    : 1003.95
    > Off-Pre : 45.62
    > On-Pre  : 19.55

❌ FULL MISSING LIST (43 subjects):
  [3, 4, 5, 8, 12, 13, 14, 17, 18, 22, 24, 25, 27, 31, 32, 34, 35, 37, 39, 40, 41, 42, 49, 50, 52, 54, 57, 61, 65, 67, 74, 76, 81, 82, 84, 88, 89, 94, 99, 101, 104, 105, 116]
Loaded cache with 7790 slices.

Preparing CHH dataset (Forced 6-dim alignment) 
Pre-flight check: Validating CHH CSV mapping...
--- CHH CSV RAW MEANS ---
  > Ag

## Basis

Fourier magnitude, because it is the only Laplacian eigenbasis whose readout is
translation invariant: a shift changes the phase, not the magnitude. DCT diagonalizes the
Laplacian under Neumann boundaries and admits no such readout. Modes are ordered by true
Fourier frequency, keeping one of each conjugate pair, since a real volume has
|F(k)| = |F(N-k)| exactly.

Magnitudes are log compressed because their variance scales with their mean. The exponent
is the slope of log-variance on log-mean across features; by the delta method x^(1-p/2)
stabilizes it, so p near 2 calls for a log. It is fit with no labels, so the choice
precedes any modeling.

In [2]:
from basis_fix import low_modes
import dbs_3d.method as method

SHAPE = (20, 20, 16)
print('first 8 of 16 modes:', low_modes(16, SHAPE, 'lap')[:8])

G['MODEL'], G['K'] = 'SpectralViT', 16
exec(compile(TAIL, 'sc_worker_tail', 'exec'), G)
print('spectral features  MSW', G['XA'].shape, ' CHH', G['XB'].shape)
print('clinical covariates', G['Ca'].shape[1])

from scipy.fft import fftn
raw = lambda V: np.stack([np.abs(fftn(V, axes=(1, 2, 3)))[:, a, b, c] for a, b, c in G['IDX']], 1)
RAW = np.hstack([raw(G['HA']), raw(G['LA'])])          # magnitudes BEFORE the log
mu, va = RAW.mean(0), RAW.var(0)
ok = (mu > 0) & (va > 0)
sl = np.polyfit(np.log(mu[ok]), np.log(va[ok]), 1)[0]
rr = np.corrcoef(np.log(mu[ok]), np.log(va[ok]))[0, 1]
print()
print(f'mean-variance slope {sl:.2f} (R^2 {rr**2:.3f}) on the raw magnitudes -- near 2 implies log')
print(f'method.variance_exponent {method.variance_exponent(RAW):.3f}  (label-free)')

first 8 of 16 modes: [(0, 0, 0), (0, 1, 0), (1, 0, 0), (0, 0, 1), (1, 1, 0), (0, 1, 1), (1, 0, 1), (1, 1, 1)]
spectral features  MSW (66, 32)  CHH (37, 32)
clinical covariates 7

mean-variance slope 1.56 (R^2 0.929) on the raw magnitudes -- near 2 implies log
method.variance_exponent 1.556  (label-free)


## Architecture

The analysis workers use `rank_model.RankAttn`. The repo class `networks.SpectralViT`
reproduces it exactly under the DBS flags, so one architecture covers both this result and
the IXI one. Verified here rather than asserted.

In [3]:
from rank_model import RankAttn
ns_, nc_ = G['ZA'].shape[1], G['DA'].shape[1]
torch.manual_seed(0); ra = RankAttn(ns_, nc_, np.resize(G['LAM'], ns_), weight='none')
torch.manual_seed(0); sv = method.spectral_vit(ns_, nc_)
ra.eval(); sv.eval()
torch.manual_seed(1); Xs, Cs = torch.randn(9, ns_), torch.randn(9, nc_)
with torch.no_grad():
    d = (ra(Xs, Cs) - sv(torch.cat([Xs, Cs], 1))).abs().max().item()
pa = sum(p.numel() for p in ra.parameters()); pb = sum(p.numel() for p in sv.parameters())
print(f'RankAttn {pa} params | networks.SpectralViT {pb} params | forward maxdiff {d}')
assert pa == pb and d == 0.0

RankAttn 2937 params | networks.SpectralViT 2937 params | forward maxdiff 0.0


## Leave-one-out over the capacity grid

Every model at every grid point, ten seeds each. Internal scores are leave-one-out over
MSW; external scores come from a model refit on all of MSW. Scores are averaged over
seeds, then thresholded once.

First slow stage; cached to `docs/dbs_3d/dbs_inference_cells.pkl`.

In [4]:
CACHE_F = os.path.join(REPO, 'docs', 'dbs_3d', 'dbs_inference_cells.pkl')
CELL = pickle.load(open(CACHE_F, 'rb')) if os.path.exists(CACHE_F) else {}
todo = [(m, K) for K in KGRID for m in MODELS if (m, K) not in CELL]
print(f'{len(CELL)} cells cached, {len(todo)} to compute')
if todo:
    t0 = time.time()
    for m, K in todo:
        G['MODEL'], G['K'] = m, K
        exec(compile(TAIL, 'sc_worker_tail', 'exec'), G)
        PR = [G['loo'](bA, sd) for sd in range(NSEED)]
        PB = [G['predict'](np.arange(len(bA)), 'EXT', bA, sd) for sd in range(NSEED)]
        CELL[(m, K)] = dict(pr=np.mean(PR, 0), pb=np.mean(PB, 0),
                            per_seed_int=np.array([roc_auc_score(bA, p) for p in PR]),
                            per_seed_ext=np.array([roc_auc_score(bB, p) for p in PB]))
        c = CELL[(m, K)]
        print(f"{m:14s} K={K:2d}  int {roc_auc_score(bA, c['pr']):.3f}  "
              f"ext {roc_auc_score(bB, c['pb']):.3f}  ({time.time()-t0:.0f}s)", flush=True)
    pickle.dump(CELL, open(CACHE_F, 'wb'))
    print()
    print(f'{len(todo)} cells computed in {time.time()-t0:.0f}s; {len(CELL)} total')

18 cells cached, 0 to compute


### What the grid actually varies

The grid index K means different things per model, so it is reported as the capacity
each model actually selects:

- **SpectralViT, Spectral LR, Spectral MLP** — K is the number of Fourier modes per nucleus.
- **Spatial ViT** — K only indexes patch size (`{8:8, 16:5, 32:4}` voxels, about 16 / 96 /
  200 tokens). It uses no spectral modes.
- **Attention U-Net** — the repo `networks.AttentionUNet` on the raw crops, both nuclei as
  input channels; K indexes base channels (4 / 7 / 12), not modes.
- **Clinical** — fits the covariates alone, so K never enters. Its grid points are
  identical and any argmax over them is an arbitrary tie-break, not a selection.

A model whose grid points are identical has not searched anything, which matters for any
multiplicity correction applied later.

In [5]:
PS_OF = {8: 8, 16: 5, 32: 4}      # Spatial ViT patch size
BC_OF = {8: 4, 16: 7, 32: 12}     # Attention U-Net base channels
SEL, DISTINCT = {}, {}
for m in MODELS:
    SEL[m] = max(KGRID, key=lambda K: CELL[(m, K)]['per_seed_int'].mean())
    DISTINCT[m] = len({round(CELL[(m, K)]['per_seed_int'].mean(), 6) for K in KGRID})

def capacity(m):
    """What this model actually selected, in its own units."""
    if m == 'Clinical':    return '--'
    if m == 'Spatial ViT': return f'patch {PS_OF[SEL[m]]}'
    if m == 'Attention UNet': return f'{BC_OF[SEL[m]]} ch'
    return f'{SEL[m]} modes'

for m in MODELS:
    row = "  ".join(f"K={K} {CELL[(m,K)]['per_seed_int'].mean():.3f}" for K in KGRID)
    note = '' if DISTINCT[m] > 1 else '   (grid points identical -- no search)'
    print(f"{m:14s} {row}   -> {capacity(m)}{note}")

SpectralViT    K=8 0.714  K=16 0.731  K=32 0.651   -> 16 modes
Spectral LR    K=8 0.567  K=16 0.692  K=32 0.541   -> 16 modes
Spectral MLP   K=8 0.555  K=16 0.542  K=32 0.545   -> 8 modes
Clinical       K=8 0.708  K=16 0.708  K=32 0.708   -> --   (grid points identical -- no search)
Spatial ViT    K=8 0.371  K=16 0.221  K=32 0.287   -> patch 8
Attention UNet K=8 0.409  K=16 0.339  K=32 0.409   -> 12 ch


## Operating point

The threshold maximizes balanced accuracy on MSW and is never fit on CHH. Applied
directly to CHH it lands in the wrong place, because the two score distributions differ in
location and spread. The *quantile* of that threshold transfers instead: it uses no CHH
labels and assumes only that the cohorts have similar non-responder prevalence
(5/66 = 7.6% vs 3/37 = 8.1%). Both operating points are reported.

In [6]:
def cal(y, p):
    """Threshold maximizing balanced accuracy."""
    best, bt = -1, float(np.median(p))
    for t in np.unique(np.concatenate([p, np.linspace(p.min(), p.max(), 300)])):
        b = balanced_accuracy_score(y, (p >= t).astype(int))
        if b > best: best, bt = b, float(t)
    return bt


def full(y, p, thr):
    """All metrics at one threshold, plus the confusion counts."""
    yh = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return dict(AUC=roc_auc_score(y, p), Acc=accuracy_score(y, yh),
                BalAcc=balanced_accuracy_score(y, yh),
                Spec=tn/(tn+fp) if (tn+fp) else np.nan,
                PPV=tp/(tp+fp) if (tp+fp) else np.nan,
                NPV=tn/(tn+fn) if (tn+fn) else np.nan,
                F1=f1_score(y, yh, zero_division=0),
                TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))


rows, OP = [], {}
for m in MODELS:
    c = CELL[(m, SEL[m])]; pr, pb = c['pr'], c['pb']
    thr = cal(bA, pr)                       # MSW only
    q = float((pr < thr).mean())            # its quantile in MSW
    tq = float(np.quantile(pb, q))          # the same quantile in CHH
    OP[m] = dict(thr=thr, q=q, thr_chh=tq,
                 shift=float((pb.mean()-pr.mean())/(pr.std()+1e-12)))
    for part, y, p, t in [('int', bA, pr, thr), ('ext raw', bB, pb, thr),
                          ('ext qtl', bB, pb, tq)]:
        rows.append(dict(model=m, capacity=capacity(m), part=part, **full(y, p, t)))
tab = pd.DataFrame(rows)
print(pd.DataFrame(OP).T.round(4).to_string())

                   thr       q  thr_chh   shift
SpectralViT     0.9924  0.4697   0.9993  0.3309
Spectral LR     0.9856  0.5303   0.9992  0.4616
Spectral MLP    0.9711  0.3636   0.9926  0.3147
Clinical        0.8548  0.5909   0.9794  0.4039
Spatial ViT     0.6332  0.0000   0.7631  0.2775
Attention UNet  0.9999  0.9697   0.9737 -0.2254


## The table

External cohort at the transferred operating point. Sensitivity omitted. `TN/3` counts
correctly identified non-responders out of the three in CHH.

In [7]:
ext = tab[tab.part == 'ext qtl'].set_index('model').loc[MODELS]
show = ext[['AUC', 'Acc', 'BalAcc', 'Spec', 'F1']].round(3)
show.insert(0, 'capacity', [capacity(m) for m in MODELS])
show['TN/3'] = ext['TN'].astype(int).astype(str) + '/3'
print(show.to_string())

cmp = pd.DataFrame({'internal BalAcc': tab[tab.part == 'int'].set_index('model')['BalAcc'],
                    'external BalAcc': ext['BalAcc']}).loc[MODELS].round(3)
cmp['gap'] = (cmp['external BalAcc'] - cmp['internal BalAcc']).round(3)
print()
print(cmp.to_string())

                capacity    AUC    Acc  BalAcc   Spec     F1 TN/3
model                                                            
SpectralViT     16 modes  0.814  0.622   0.794  1.000  0.741  3/3
Spectral LR     16 modes  0.765  0.486   0.569  0.667  0.627  2/3
Spectral MLP     8 modes  0.745  0.649   0.657  0.667  0.772  2/3
Clinical              --  0.667  0.432   0.539  0.667  0.571  2/3
Spatial ViT      patch 8  0.618  0.919   0.500  0.000  0.958  0/3
Attention UNet     12 ch  0.392  0.135   0.529  1.000  0.111  3/3

                internal BalAcc  external BalAcc    gap
model                                                  
SpectralViT               0.787            0.794  0.007
Spectral LR               0.754            0.569 -0.185
Spectral MLP              0.628            0.657  0.029
Clinical                  0.721            0.539 -0.182
Spatial ViT               0.500            0.500  0.000
Attention UNet            0.516            0.529  0.013


### Against the standalone pipeline

`docs/dbs_3d/metrics2.log` is written by `run_metrics.sh`, which drives `metrics2.py` at
the capacities `select_capacity.py` derives. That is a separate implementation of the same
protocol, so agreement with the table above cross-checks two code paths rather than
comparing the notebook with itself. Every row must match to the digit.

In [8]:
LOG = os.path.join(REPO, 'docs', 'dbs_3d', 'metrics2.log')
if not os.path.exists(LOG):
    print('Patient data missing -- docs/ is gitignored, so the recorded run is absent from a clean clone.')
    print(f'  expected {LOG}')
    print('  the table above is still fully recomputed; only this cross-check is skipped.')
else:
    pat = re.compile(r'^(.{20}) +(int|ext raw|ext qtl) +AUC ([\d.]+) Acc ([\d.]+) BalAcc ([\d.]+) '
                     r'Spec ([\d.]+) PPV (nan|[\d.]+) NPV (nan|[\d.]+) F1 ([\d.]+) \| TN (\d+) FP (\d+)')
    same, diffs = 0, []
    for ln in open(LOG).read().splitlines():
        g = pat.match(ln)
        if not g or g.group(1).strip() not in MODELS: continue
        m, part = g.group(1).strip(), g.group(2)
        r = tab[(tab.model == m) & (tab.part == part)].iloc[0]
        bad = [(k, float(g.group(j)), round(float(r[k]), 3))
               for k, j in [('AUC', 3), ('Acc', 4), ('BalAcc', 5), ('Spec', 6), ('F1', 9), ('TN', 10)]
               if abs(round(float(r[k]), 3) - float(g.group(j))) > 5e-4]
        if bad: diffs.append((m, part, bad))
        else: same += 1
    want = 3 * len(MODELS)
    print(f'{same} rows identical, {len(diffs)} differing, of {want} expected')
    if same + len(diffs) != want:
        print(f'INCOMPLETE: only {same + len(diffs)} of {want} rows parsed -- '
              'the rest were skipped, not verified')
    elif diffs:
        print(f'DIFFERENCES: {diffs}')
    else:
        print('ALL ROWS IDENTICAL')

18 rows identical, 0 differing, of 18 expected
ALL ROWS IDENTICAL


## External validation — the pre-specified cell

The cell chosen on internal CV, read on the external set once, with its permutation p and
a bootstrap CI. The two disagree by construction at three non-responders; both are shown.

In [9]:
rng = np.random.RandomState(0)
n, n1 = len(bB), int(bB.sum()); n0 = n - n1
for m in MODELS:
    pb = CELL[(m, SEL[m])]['pb']
    a = roc_auc_score(bB, pb)
    null = np.array([roc_auc_score(bB[rng.permutation(n)], pb) for _ in range(20000)])
    boot = [roc_auc_score(bB[i], pb[i]) for i in rng.randint(0, n, (5000, n))
            if len(np.unique(bB[i])) > 1]
    print(f"{m:14s} {capacity(m):10s} ext AUC {a:.3f}  perm p {(null>=a).mean():.4f}  "
          f"95% CI [{np.percentile(boot,2.5):.3f}, {np.percentile(boot,97.5):.3f}]")

SpectralViT    16 modes   ext AUC 0.814  perm p 0.0428  95% CI [0.667, 0.939]
Spectral LR    16 modes   ext AUC 0.765  perm p 0.0656  95% CI [0.424, 0.972]
Spectral MLP   8 modes    ext AUC 0.745  perm p 0.0928  95% CI [0.197, 1.000]
Clinical       --         ext AUC 0.667  perm p 0.1852  95% CI [0.324, 0.944]
Spatial ViT    patch 8    ext AUC 0.618  perm p 0.2712  95% CI [0.343, 0.944]
Attention UNet 12 ch      ext AUC 0.392  perm p 0.7255  95% CI [0.028, 0.972]


## Seed stability

Independent initializations of the same configuration. Rules out an initialization lottery.

In [10]:
for m in MODELS:
    c = CELL[(m, SEL[m])]
    i_, e_ = c['per_seed_int'], c['per_seed_ext']
    print(f"{m:14s} internal {i_.mean():.3f}+-{i_.std():.3f} [{i_.min():.3f}, {i_.max():.3f}]   "
          f"external {e_.mean():.3f}+-{e_.std():.3f} [{e_.min():.3f}, {e_.max():.3f}]   "
          f"ext<0.5 in {(e_<0.5).sum()}/{len(e_)}")

SpectralViT    internal 0.731+-0.061 [0.652, 0.866]   external 0.841+-0.066 [0.735, 0.951]   ext<0.5 in 0/10
Spectral LR    internal 0.692+-0.000 [0.692, 0.692]   external 0.765+-0.000 [0.765, 0.765]   ext<0.5 in 0/10
Spectral MLP   internal 0.555+-0.066 [0.416, 0.643]   external 0.767+-0.050 [0.706, 0.853]   ext<0.5 in 0/10
Clinical       internal 0.708+-0.000 [0.708, 0.708]   external 0.667+-0.000 [0.667, 0.667]   ext<0.5 in 0/10
Spatial ViT    internal 0.371+-0.101 [0.220, 0.502]   external 0.690+-0.121 [0.510, 0.912]   ext<0.5 in 0/10
Attention UNet internal 0.409+-0.062 [0.315, 0.551]   external 0.602+-0.103 [0.373, 0.755]   ext<0.5 in 1/10


## Imaging ablation

Train normally, then degrade one input stream **at test time only**. `zero` hands the model
a constant it can partly ignore; `shuffle` hands it another patient's values, preserving the
marginal distribution. A model that ignores a stream is unaffected by both.

In [11]:
import torch.nn as nn
G['MODEL'], G['K'] = 'SpectralViT', SEL['SpectralViT']
exec(compile(TAIL, 'sc_worker_tail', 'exec'), G)
ZA, ZB, DA, DB, LAM, dev, T, sig = (G['ZA'], G['ZB'], G['DA'], G['DB'], G['LAM'],
                                    G['dev'], G['T'], G['sig'])
abl = {k: [] for k in ['intact', 'img zero', 'img shuffle', 'clin zero', 'clin shuffle']}
for sd in range(5):
    torch.manual_seed(sd)
    mo = RankAttn(ZA.shape[1], DA.shape[1], np.resize(LAM, ZA.shape[1]), weight='none').to(dev)
    opt = torch.optim.AdamW(mo.parameters(), 3e-3, weight_decay=0.1); lf = nn.BCEWithLogitsLoss()
    xt, ct = T(ZA), T(DA); yt = torch.tensor(bA.astype(float), dtype=torch.float32, device=dev)
    for _ in range(200):
        opt.zero_grad(); lf(mo(xt, ct), yt).backward(); opt.step()
    mo.eval(); rr2 = np.random.RandomState(sd)
    with torch.no_grad():
        a = lambda Z, D: roc_auc_score(bB, sig(mo(T(Z), T(D)).cpu().numpy()))
        abl['intact'].append(a(ZB, DB))
        abl['img zero'].append(a(np.zeros_like(ZB), DB))
        abl['img shuffle'].append(a(ZB[rr2.permutation(len(ZB))], DB))
        abl['clin zero'].append(a(ZB, np.zeros_like(DB)))
        abl['clin shuffle'].append(a(ZB, DB[rr2.permutation(len(DB))]))
abl = {k: np.array(v) for k, v in abl.items()}
for k, v in abl.items():
    print(f"{k:14s} {v.mean():.3f} +- {v.std():.3f}   (delta {v.mean()-abl['intact'].mean():+.3f})")

intact         0.853 +- 0.073   (delta +0.000)
img zero       0.761 +- 0.043   (delta -0.092)
img shuffle    0.667 +- 0.152   (delta -0.186)
clin zero      0.788 +- 0.074   (delta -0.065)
clin shuffle   0.718 +- 0.108   (delta -0.135)


## Summary

In [12]:
S = pd.DataFrame({
    'capacity': {m: capacity(m) for m in MODELS},
    'int LOO AUC': {m: round(CELL[(m, SEL[m])]['per_seed_int'].mean(), 3) for m in MODELS},
    'ext AUC': {m: round(tab[(tab.model == m) & (tab.part == 'ext qtl')]['AUC'].iloc[0], 3) for m in MODELS},
    'ext BalAcc': {m: round(tab[(tab.model == m) & (tab.part == 'ext qtl')]['BalAcc'].iloc[0], 3) for m in MODELS},
    'ext Spec': {m: round(tab[(tab.model == m) & (tab.part == 'ext qtl')]['Spec'].iloc[0], 3) for m in MODELS},
    'TN/3': {m: f"{int(tab[(tab.model == m) & (tab.part == 'ext qtl')]['TN'].iloc[0])}/3" for m in MODELS},
}).loc[MODELS]
print(S.to_string())

tab.round(4).to_csv(os.path.join(REPO, 'docs', 'dbs_3d', 'dbs_inference_table.csv'), index=False)
S.to_csv(os.path.join(REPO, 'docs', 'dbs_3d', 'dbs_inference_summary.csv'))
print()
print('wrote docs/dbs_3d/dbs_inference_table.csv and dbs_inference_summary.csv')

                capacity  int LOO AUC  ext AUC  ext BalAcc  ext Spec TN/3
SpectralViT     16 modes        0.731    0.814       0.794     1.000  3/3
Spectral LR     16 modes        0.692    0.765       0.569     0.667  2/3
Spectral MLP     8 modes        0.555    0.745       0.657     0.667  2/3
Clinical              --        0.708    0.667       0.539     0.667  2/3
Spatial ViT      patch 8        0.371    0.618       0.500     0.000  0/3
Attention UNet     12 ch        0.409    0.392       0.529     1.000  3/3

wrote docs/dbs_3d/dbs_inference_table.csv and dbs_inference_summary.csv


**Power ceiling.** CHH has 3 non-responders x 34 responders = 102 discordant pairs, so
external AUC is quantized in steps of 1/102 and specificity in steps of 1/3; the internal
permutation null rests on 5 non-responders. No change to the model moves either floor.
Only more non-responders does.